# Topical Lectures, Controls
## Real drone, part 1: connecting to your drone

Andreas Freise, 30.10.2026

Now we move from the simulator to the real Crazyflie. In this notebook we make and test the basic connection using the radio dongle and talk to the drone via Python.

The laptop talks to the drone through the Crazyradio dongle, and every drone answers to its own radio address: a channel, a data rate and a five-byte address. In the course the data rate is the factory value, 2M, and the last byte of the address is the label on the sticker of the drone, a hex digit from 0 to B. Each drone has its own radio channel, 2 + 7 times its label, so that the drones in the room do not share a frequency; channel n lies at 2400 + n MHz. The drone with label 2 therefore has the address `E7E7E7E702` on channel 16, and the full URI, as the radio software wants it, is

```
radio://0/16/2M/E7E7E7E702
```

A wrong address commands another group's drone. So before any control loop we find our drone, make it show itself and read what it reports. This notebook does not arm the motors.

### Before you start

* Switch the drone on (the blue LEDs light up) and put it on the desk, in its protective frame.
* Plug the Crazyradio dongle into the laptop.
* Only one program can talk to a drone at a time: close cfclient before running the cells.


In [7]:
import crazyflie

## Your drone

Enter the label of your drone as it reads on the sticker, `LABEL = 2`; a letter goes in quotes, `LABEL = 'A'`. For sticker 0, use `LABEL = 0`: its address is `E7E7E7E700` on channel 2. A drone that still has the factory address `E7E7E7E7E7` has `E7` as its last byte, so `LABEL = 'E7'`.


In [8]:
LABEL = 4
URI = crazyflie.drone_uri(LABEL)

## Can you connect to your drone?

The next cell looks for your drone on all radio channels and prints the line it answers to. It takes a fraction of a second. Without a warning, the drone sits on the channel of its label and the URI above reaches it.

If nothing answers: is the drone switched on (blue LEDs), is the dongle plugged in, is the label right? `crazyflie.find_drones()` without a label looks for all labels of the course on all channels, which takes a few seconds, and lists every drone in the room with the channel it is on, the drones of the other groups included. `crazyflie.find_drones(channel=80)` asks one channel only.


In [9]:
crazyflie.find_drones(LABEL)

drones in range (label, URI, what to enter in the next cell):
  label 4    radio://0/30/2M/E7E7E7E704   LABEL = 4


{4: 'radio://0/30/2M/E7E7E7E704'}

If the above prints a warning, the radio of the drone is not set as the course expects, for example a drone that still sits on the factory channel 80, and the URI above does not reach it. Do not go on with the work but ask an instructor, who will then change the drone settings to the correct channel and restart it.

## Connect and check

The next cell opens the radio link, prints the status of the drone and makes its LEDs blink three times. Watch your drone while the cell runs: if a drone on another desk blinks instead, the label and the address do not match and we need to know.

The status lists the battery voltage (fly only above 3.5 V at rest, the status says so; charge over USB when it is lower), whether the flow deck is detected (the deck with the two small windows on the underside, which measures height and motion) and the flags of the supervisor, the part of the firmware that decides whether the drone may be armed. The last two lines put the height estimate next to the reading of the height sensor and watch the position estimate for two seconds. A warning there means the deck has stopped delivering, which it does now and then without any other sign: switch the drone off and on and connect again.


In [6]:
with crazyflie.connect(URI) as scf:
    crazyflie.status(scf)
    crazyflie.identify(scf)

Drone status of radio://0/30/2M/E7E7E7E704
  battery      : charging, 3.97 V at rest, 64 % of a full charge
  flight       : ok
  flow deck    : attached
  can be armed : no
  is armed     : no
  is tumbled   : no
  is locked    : no
  height       : estimate -0.003 m, sensor 0.009 m, agree
  position     : estimate moved 0.011 m in 2 s on the ground, ok
the four LEDs of your drone blink 3 times


## What the drone measures

The flight notebooks hold the height with the downward range sensor of the flow deck. The next cell prints its reading twice a second for eight seconds. Lift the drone slowly to about 30 cm and put it back: the height follows your hand. The second number is the height estimate of the firmware, which combines the sensor with the accelerometer. At the end the cell says whether the estimate followed the sensor.


In [10]:
with crazyflie.connect(URI) as scf:
    crazyflie.watch_height(scf, duration=8)

lift the drone slowly and put it back, 8 s
  t =  0.5 s   height sensor  0.01 m   estimated z -0.00 m
  t =  1.0 s   height sensor  0.01 m   estimated z -0.01 m
  t =  1.5 s   height sensor  0.01 m   estimated z -0.01 m
  t =  2.0 s   height sensor  0.01 m   estimated z  0.01 m
  t =  2.5 s   height sensor  0.01 m   estimated z -0.00 m
  t =  3.0 s   height sensor  0.01 m   estimated z -0.00 m
  t =  3.5 s   height sensor  0.01 m   estimated z -0.00 m
  t =  4.0 s   height sensor  0.01 m   estimated z -0.01 m
  t =  4.5 s   height sensor  0.01 m   estimated z -0.00 m
  t =  5.0 s   height sensor  0.01 m   estimated z  0.02 m
  t =  5.5 s   height sensor  0.01 m   estimated z -0.01 m
  t =  6.0 s   height sensor  0.01 m   estimated z -0.00 m
  t =  6.5 s   height sensor  0.01 m   estimated z -0.00 m
  t =  7.1 s   height sensor  0.01 m   estimated z -0.00 m
  t =  7.6 s   height sensor  0.01 m   estimated z  0.00 m
  t =  8.1 s   height sensor  0.01 m   estimated z  0.00 m
the estimate 

## The next notebooks

Write the label of your drone on your group sheet. Every flight notebook starts with the same line, `LABEL = ...`, and builds the URI from it.